# Bringing the Heat 🔥
## Notebook → reproducible artifact → release decision
**Muntaser Syed · Miami Dade College**

Our application is a news router: **World / Sports / Business / Sci/Tech**.
One small classifier connects the whole stack: Hub, Datasets, Transformers, PEFT, Accelerate, Optimum, Gradio.

**Before the talk:** run `setup.ps1`, then `rehearse.ps1` from this folder. This notebook is the live driver over prepared local artifacts. Package downloads and full training happen before the session.

**Live rule:** narrate a running operation for 15 seconds; if it is still busy, move to the saved evidence. Never make the audience wait for a download.


In [1]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'demo.py').exists():
    ROOT = ROOT / 'demo'
assert (ROOT / 'demo.py').exists(), 'Open this notebook from the demo folder.'
os.chdir(ROOT)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

def read(path):
    return json.loads((ROOT / path).read_text(encoding='utf-8'))

def run(*args, expected=0):
    result = subprocess.run([sys.executable, 'demo.py', *args], capture_output=True, text=True)
    print(result.stdout)
    allowed = (expected,) if isinstance(expected, int) else expected
    if result.returncode not in allowed:
        print(result.stderr)
        raise RuntimeError(f'Expected exit {expected}, got {result.returncode}')
    return result

run('preflight');


{
  "status": "PASS",
  "environment": {
    "python": "3.12.12",
    "platform": "Windows-11-10.0.26200-SP0",
    "processor": "Intel64 Family 6 Model 183 Stepping 1, GenuineIntel",
    "cpu_logical_count": 32,
    "cuda_available": true,
    "gpu": "NVIDIA GeForce RTX 4090 Laptop GPU",
    "packages": {
      "torch": "2.8.0+cu128",
      "transformers": "4.55.4",
      "peft": "0.17.1",
      "accelerate": "1.10.1",
      "datasets": "4.0.0",
      "optimum-onnx": "0.0.3",
      "onnxruntime": "1.30.0",
      "scikit-learn": "1.7.2",
      "gradio": "5.50.0"
    }
  },
  "split_integrity": "PASS",
  "cpu_threads_for_benchmark": 4,
  "cuda_smoke": 32.0
}



## 1 · Hub objects become reproducible inputs
The Hub holds more than model weights: revisions, dataset schemas, cards, licenses, adapters, and runnable demos all matter.

`from_pretrained()` gets us started. A **commit SHA + environment lock + immutable split** makes the experiment explainable later.

This is a *balanced teaching subset*. Validation comes from the official training split; the locked test comes from the official test split. We remove normalized exact duplicates across all three; near duplicates require additional checks.


In [2]:
manifest = read('artifacts/manifest.json')
display(pd.DataFrame(manifest['splits']).T[['count', 'sha256']])
print('Model:', manifest['model_id'], '@', manifest['model_revision'])
print('Dataset:', manifest['dataset_id'], '@', manifest['dataset_revision'])
print('Class order:', manifest['labels'])


,count,sha256
train,3200,2a00fb6636552032d2835add36a18aaf2e91d1a0ac38a5...
validation,400,1d1085b9982a61a1081283e67e0410a1a5ca5533fc0b1e...
test,800,05b61847a83e142493815794a7590b09556b770e4d3dd0...


Model: distilbert/distilbert-base-uncased @ 12040accade4e8a0f71eabdb258fecc2e7e948be
Dataset: fancyzhx/ag_news @ eb185aade064a813bc0b7f42de02595523103ca4
Class order: ['World', 'Sports', 'Business', 'Sci/Tech']


## 2 · Change the small part: PEFT + Accelerate
LoRA learns low-rank updates to attention projections while most of the base model stays frozen. Our new classification head also trains.

```python
config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=['q_lin', 'v_lin'],
    modules_to_save=['pre_classifier', 'classifier'],
)
model = get_peft_model(base, config)
model, optimizer, train_loader, validation_loader = accelerator.prepare(
    model, optimizer, train_loader, validation_loader
)
loss = model(**batch).loss
accelerator.backward(loss)
optimizer.step()
```

**Division of labor:** PEFT controls *what learns*; Accelerate controls *where and how the loop runs*. DDP replicates the model across GPUs. `device_map='auto'` is a different inference-placement tool, not this distributed training recipe.


In [3]:
# Optional real, isolated five-step training demo. Keep False for instant replay.
RUN_TRAINING_SMOKE = False
if RUN_TRAINING_SMOKE:
    run('train', '--smoke')  # writes smoke_adapter; preserves rehearsed adapter

training = read('results/training.json')
print(f"Trainable: {training['trainable_parameters']:,} / {training['total_parameters']:,}")
print(f"Trainable fraction: {training['trainable_percent']:.2f}%")
display(pd.DataFrame([{'epoch': row['epoch'], 'loss': row['mean_loss'],
    'validation_macro_f1': row['validation']['macro_f1']} for row in training['history']]))
print('Checkpoint chosen using validation only.')


Trainable: 741,124 / 67,697,672
Trainable fraction: 1.09%


,epoch,loss,validation_macro_f1
0,1,0.494572,0.906663
1,2,0.280934,0.905205
2,3,0.245168,0.907437


Checkpoint chosen using validation only.


## 3 · Evaluate the candidate, then evaluate the exported artifact
Accuracy can hide a weak class. Macro F1 gives each class equal weight; per-class recall answers *which category are we missing?*

We compare three implementations on exactly the same 800 examples. A majority-class baseline makes “better” meaningful. Read the confidence interval as uncertainty on this sample, not a promise about tomorrow's news.


In [4]:
evaluation = read('results/evaluation.json')
summary = {name: {key: scores[key] for key in ['n', 'accuracy', 'macro_f1']}
           for name, scores in evaluation['models'].items()}
summary['majority_baseline'] = {key: evaluation['majority_class_baseline'][key]
                               for key in ['n', 'accuracy', 'macro_f1']}
display(pd.DataFrame(summary).T)
display(pd.DataFrame(evaluation['models']['int8']['per_class']).T)
print('INT8 accuracy 95% Wilson interval:', evaluation['models']['int8']['accuracy_95pct_wilson'])


,n,accuracy,macro_f1
pytorch,800.0,0.9025,0.902308
onnx,800.0,0.9025,0.902308
int8,800.0,0.9050,0.904809
majority_baseline,800.0,0.2500,0.100000


,precision,recall,f1,support
World,0.925134,0.865,0.894057,200.0
Sports,0.951456,0.980,0.965517,200.0
Business,0.857143,0.900,0.878049,200.0
Sci/Tech,0.888325,0.875,0.881612,200.0


INT8 accuracy 95% Wilson interval: [0.8827021384517765, 0.9234268302405629]


## 4 · Optimum: export and measure on the target
The adapter is first **merged into the base** to produce a standalone model, then exported to ONNX. Dynamic INT8 quantization targets this laptop's AVX2 CPU.

```python
ort_model = ORTModelForSequenceClassification.from_pretrained('artifacts/merged', export=True)
quantizer = ORTQuantizer.from_pretrained('artifacts/onnx_fp32')
quantizer.quantize(save_dir='artifacts/onnx_int8',
    quantization_config=AutoQuantizationConfig.avx2(is_static=False, per_channel=False))
```

Do not compare GPU batch throughput to CPU single-request latency. Here all three backends use **CPU, four threads, batch 1, fixed 128 tokens, 10 warmups, 100 measured requests**. Time includes tokenization + forward; excludes loading, network and queueing. Weight bytes are file size, not peak RAM.


In [5]:
benchmark = read('results/benchmark.json')
display(pd.DataFrame({name: {'p50_ms': value['p50_ms'], 'p95_ms': value['p95_ms'],
    'sequential_requests_per_second': value['sequential_requests_per_second'],
    'weights_MiB': value['weights_bytes'] / 1024**2}
    for name, value in benchmark['models'].items()}).T.round(2))
print(benchmark['environment']['platform'])
print(benchmark['timing_scope'])


,p50_ms,p95_ms,sequential_requests_per_second,weights_MiB
pytorch,33.48,59.81,26.74,255.43
onnx,30.33,41.56,31.84,255.53
int8,25.82,33.57,37.33,64.25


Windows-11-10.0.26200-SP0
Tokenization + model forward, sequential batch=1 requests; excludes load, networking and queueing


## 5 · Make “ship it” executable
Our *teaching* policy was written before the test: minimum quality, every-class recall, limited quantization regression, a CPU p95 budget, and a footprint budget. Real product owners set their own values from costs, risks and service goals.

First pass the real candidate. Then inject a simulated Business-recall regression. The overall accuracy is unchanged in this deliberate demonstration, but the gate must still block release.


In [6]:
run('gate', expected=(0, 2))  # A genuine block is a valid demonstration outcome.
run('gate', '--inject-failure', expected=2);


{
  "passed": true,
  "checks": {
    "same_heldout_split": true,
    "exact_runtime_artifacts": true,
    "complete_class_report": true,
    "benchmark_protocol": true,
    "minimum_test_size": true,
    "accuracy": true,
    "macro_f1": true,
    "every_class_recall": true,
    "quantization_f1_regression": true,
    "cpu_p95_latency": true,
    "weight_size": true
  },
  "policy": {
    "min_test_examples": 800,
    "min_accuracy": 0.8,
    "min_macro_f1": 0.8,
    "min_class_recall": 0.65,
    "max_macro_f1_drop": 0.02,
    "max_cpu_p95_ms": 75.0,
    "max_weights_bytes": 104857600
  },
  "injected_failure": false,
  "scope": "Teaching gate on this locked sample and CPU; not a production certification"
}



{
  "passed": false,
  "checks": {
    "same_heldout_split": true,
    "exact_runtime_artifacts": true,
    "complete_class_report": true,
    "benchmark_protocol": true,
    "minimum_test_size": true,
    "accuracy": true,
    "macro_f1": true,
    "every_class_recall": false,
    "quantization_f1_regression": true,
    "cpu_p95_latency": true,
    "weight_size": true
  },
  "policy": {
    "min_test_examples": 800,
    "min_accuracy": 0.8,
    "min_macro_f1": 0.8,
    "min_class_recall": 0.65,
    "max_macro_f1_drop": 0.02,
    "max_cpu_p95_ms": 75.0,
    "max_weights_bytes": 104857600
  },
  "injected_failure": true,
  "scope": "Teaching gate on this locked sample and CPU; not a production certification"
}



## 6 · The exact artifact behind a local app
Try your own headline. Scores are **not calibrated confidence** and this four-way classifier has no “unknown” class. Mixed-topic text can be ambiguous.

For the Gradio UI, open a terminal in this folder and run:

```powershell
.venv/Scripts/python.exe app.py
```

Open <http://127.0.0.1:7860>. This binds to localhost; it is a preview UI, not an authenticated production service.


In [7]:
from app import classify
display(classify('The Miami team won the championship after a dramatic final quarter.'))


E:\data\phd\huggingfacetalk\demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'World': 0.0021931189112365246,
 'Sports': 0.9956372380256653,
 'Business': 0.0018130040261894464,
 'Sci/Tech': 0.00035665236646309495}

## The engineering handoff
- **Edge:** ship the measured artifact and tokenizer; remeasure on the real device. ARM64 has a different export/quantization target.
- **GPU training:** use Accelerate's launcher and config; keep effective batch size and evaluation correct as world size changes.
- **LLM serving:** separately choose a server/runtime that supports your model and hardware. Sharding to fit memory and batching for throughput are different decisions.
- **Release:** version the artifact, gate it, stage it, observe it, and keep a rollback path.

**Take-home challenge:** change LoRA rank using validation, propose a new slice metric, or benchmark a different target. Freeze the test and policy before measuring the final candidate.

[Accelerate quicktour](https://huggingface.co/docs/accelerate/quicktour) · [PEFT LoRA](https://huggingface.co/docs/peft/developer_guides/lora) · [Optimum ONNX](https://huggingface.co/docs/optimum-onnx) · [AG News](https://huggingface.co/datasets/fancyzhx/ag_news)
